# Phase 1 — kiểm định theo cặp (paired bootstrap)

**Notebook này KHÔNG chạy lại retrieval.** Không nạp model, không cần GPU, chạy vài giây.

Lý do: phép kiểm định cần thiết **đã được tính sẵn** trong lần chạy Phase 1.
`common/newsqa_rag/experiments.py::paired_comparison` chạy bootstrap theo cặp trên
từng câu hỏi cho **mọi cặp cấu hình**, và ghi kết quả vào `comparison.json` của mỗi
vòng. Chỉ có `comparison.csv` (số tổng hợp) được chép ra `docs/reports/phase1/`,
nên phần paired chưa ai đọc.

## Vì sao paired mới là phép đo đúng

Báo cáo hiện dùng **độ chồng lấn của hai khoảng CI95 biên**. Cách đó quá thận trọng:
mọi cấu hình chạy trên **đúng cùng 281 câu hỏi**, nên phần lớn phương sai là do
*câu hỏi khó hay dễ* — thứ triệt tiêu khi lấy hiệu theo từng câu.

```
  CI95 biên :  đo   mean(A)   và   mean(B)   rồi xem hai khoảng có chạm nhau không
  paired    :  đo   mean(A_i - B_i)  trên từng câu i  ->  phương sai nhỏ hơn nhiều
```

Quy tắc đọc: **khoảng CI95 của hiệu không chứa 0 ⇒ khác biệt có ý nghĩa.**

## Notebook trả lời chính xác ba câu

1. `e5-base-v2` có thật sự hơn `bge-small` không? (mục 3.2C của báo cáo)
2. `BGE-M3` có thật sự hơn `BM25-stemmed` không? (mục 3.2B)
3. Reranker, hybrid, chunk size — cái nào thật sự tách bạch?

---
## 0. Chuẩn bị

**Trên Kaggle:** Add Input → gắn output của phiên chạy Phase 1 (vòng 1 + vòng 2 + vòng 3).
Đó là nơi chứa `comparison.json` và `deterministic_scores.jsonl`.

**Ở máy:** trỏ `SEARCH_ROOTS` vào thư mục đã giải nén.


In [ ]:
REPO_URL = 'https://github.com/ThomasdeCarpio/Text-Mining---NewsQA-RAG.git'
REPO_COMMIT = 'main'  # ghim SHA khi cần tái lập chính xác

# Nơi tìm experiment dir. Trên Kaggle input thường nằm dưới /kaggle/input/<ten-dataset>/
SEARCH_ROOTS = ['/kaggle/input', '/kaggle/working', '.']

BOOTSTRAP_SAMPLES = 1000   # giữ nguyên mặc định của repo
SEED = 42                  # giữ nguyên -> kết quả tái lập đúng từng chữ số


In [ ]:
import json, sys, subprocess, itertools
from pathlib import Path

PROJECT = Path('Text-Mining---NewsQA-RAG')
if not PROJECT.exists():
    subprocess.run(['git', 'clone', '--filter=blob:none', REPO_URL, str(PROJECT)], check=True)
    if REPO_COMMIT != 'main':
        subprocess.run(['git', 'fetch', '--depth=1', 'origin', REPO_COMMIT], cwd=PROJECT, check=True)
        subprocess.run(['git', 'checkout', '--detach', REPO_COMMIT], cwd=PROJECT, check=True)
sys.path.insert(0, str(PROJECT / 'common'))

# Dùng đúng hàm của repo, không viết lại phép kiểm định.
from newsqa_rag.experiments import paired_comparison
print('paired_comparison sẵn sàng')


---
## 1. Tìm điểm số theo từng câu hỏi

Mỗi run của Phase 1 ghi `deterministic_scores.jsonl` — một dòng cho mỗi câu hỏi.
Đây là nguyên liệu của phép kiểm định theo cặp.


In [ ]:
def load_jsonl(path):
    return [json.loads(l) for l in Path(path).read_text(encoding='utf-8').splitlines() if l.strip()]

runs = {}          # run_id -> danh sách điểm theo từng câu
stored_paired = [] # paired_comparisons đã tính sẵn trong comparison.json

for root in SEARCH_ROOTS:
    root = Path(root)
    if not root.exists():
        continue
    for meta in root.rglob('experiment_run.json'):
        scores = meta.parent / 'deterministic_scores.jsonl'
        if not scores.exists():
            continue
        run_id = json.loads(meta.read_text(encoding='utf-8'))['run_id']
        runs.setdefault(run_id, load_jsonl(scores))
    for comparison in root.rglob('comparison.json'):
        blob = json.loads(comparison.read_text(encoding='utf-8'))
        for row in blob.get('paired_comparisons', []):
            stored_paired.append({'metric': blob.get('paired_metric'), **row})

print(f'{len(runs)} run có điểm theo từng câu')
print(f'{len(stored_paired)} cặp đã được tính sẵn trong comparison.json')
if not runs:
    raise SystemExit(
        'Không tìm thấy deterministic_scores.jsonl.\n'
        'Trên Kaggle: Add Input và gắn output của phiên chạy Phase 1.'
    )
for run_id in sorted(runs):
    print(f'  {run_id:58s} {len(runs[run_id]):>4d} câu')


---
## 2. Các so sánh báo cáo đang dựa vào

Khớp theo mảnh chuỗi trong `run_id`, nên không cần biết hash đuôi.


In [ ]:
# (nhãn, mảnh nhận dạng A, mảnh nhận dạng B, mục trong báo cáo)
COMPARISONS = [
    ('e5-base  vs  bge-small        (dense, resolved)',
     'resolved-development-dense', 'resolved-development-dense', '3.2C'),
    ('BGE-M3   vs  BM25-stemmed     (sparse, resolved)',
     'resolved-development-sparse', 'resolved-development-sparse', '3.2B'),
]

def pick(*fragments, exclude=()):
    """Tìm đúng một run_id chứa mọi mảnh trong fragments."""
    hits = [r for r in runs
            if all(f in r for f in fragments) and not any(e in r for e in exclude)]
    if len(hits) != 1:
        print(f'  [bỏ qua] {fragments} khớp {len(hits)} run: {hits}')
        return None
    return hits[0]

# Vòng 1 dùng index trong tên run; nếu không có, đối chiếu qua experiment_run.json
def by_index(variant, index_fragment):
    for root in SEARCH_ROOTS:
        root = Path(root)
        if not root.exists():
            continue
        for meta in root.rglob('experiment_run.json'):
            blob = json.loads(meta.read_text(encoding='utf-8'))
            params = blob.get('parameters', {})
            if params.get('variant') != variant:
                continue
            if index_fragment in str(params.get('index', '')):
                if blob['run_id'] in runs:
                    return blob['run_id']
    return None

print('Ví dụ tra cứu theo index:')
for frag in ('e5_base', 'bge_small', 'bge_m3', 'bm25_okapi_stemmed'):
    print(f'  {frag:22s} -> {by_index("resolved", frag)}')


---
## 3. Chạy kiểm định

`paired_metric` lưu sẵn chỉ là `retrieval.mrr@5`, nhưng báo cáo lập luận bằng
`nDCG@5` và `Hit@1`. Tính lại trên cùng file điểm — vẫn dùng hàm của repo.


In [ ]:
METRICS = ['retrieval.ndcg@5', 'retrieval.mrr@5', 'retrieval.hit_rate@1', 'retrieval.hit_rate@5']

def verdict(low, high):
    if low > 0:  return 'B > A  CO Y NGHIA'
    if high < 0: return 'A > B  CO Y NGHIA'
    return 'khong tach duoc (CI chua 0)'

def test(label, run_a, run_b, section=''):
    if not run_a or not run_b or run_a == run_b:
        print(f'\n{label}\n  [thieu du lieu]')
        return
    print(f'\n{label}' + (f'   (bao cao muc {section})' if section else ''))
    print(f'  A = {run_a}')
    print(f'  B = {run_b}')
    for metric in METRICS:
        out = paired_comparison(runs[run_a], runs[run_b], metric, BOOTSTRAP_SAMPLES, SEED)
        if not out:
            continue
        d, lo, hi = out['mean_delta_right_minus_left'], out['ci95_low'], out['ci95_high']
        print(f'    {metric:22s} delta(B-A) = {d:+.4f}  CI95 [{lo:+.4f}, {hi:+.4f}]  n={out["n_pairs"]}  -> {verdict(lo, hi)}')

# --- Cau hoi 1: mục 3.2C ---
test('e5-base-v2  vs  bge-small-en-v1.5   (dense thuan, resolved)',
     by_index('resolved', 'bge_small'), by_index('resolved', 'e5_base'), '3.2C')

# --- Cau hoi 2: mục 3.2B ---
test('BM25-stemmed  vs  BGE-M3   (sparse thuan, resolved)',
     by_index('resolved', 'bm25_okapi_stemmed'), by_index('resolved', 'bge_m3'), '3.2B')
test('BM25-stemmed  vs  BGE-M3   (sparse thuan, original)',
     by_index('original', 'bm25_okapi_stemmed'), by_index('original', 'bge_m3'), '3.2B')

# --- Cau hoi 3: reranker / hybrid / chunk ---
test('khong rerank  vs  bge-reranker-large   (sparse, resolved)',
     pick('resolved', 'sparse', 'noop'), pick('resolved', 'sparse', 'cross-encoder'), '4.2A')


---
## 4. Bảng tổng hợp cho báo cáo

Chạy mọi cặp có sẵn trên `nDCG@5` và in ra cặp nào tách bạch.


In [ ]:
rows = []
for a, b in itertools.combinations(sorted(runs), 2):
    out = paired_comparison(runs[a], runs[b], 'retrieval.ndcg@5', BOOTSTRAP_SAMPLES, SEED)
    if not out:
        continue
    rows.append({
        'A': a, 'B': b,
        'delta': out['mean_delta_right_minus_left'],
        'lo': out['ci95_low'], 'hi': out['ci95_high'],
        'n': out['n_pairs'],
        'significant': out['ci95_low'] > 0 or out['ci95_high'] < 0,
    })

rows.sort(key=lambda r: -abs(r['delta']))
sig = [r for r in rows if r['significant']]
print(f'{len(rows)} cap so sanh, {len(sig)} cap TACH BACH tren nDCG@5\n')
for r in rows[:40]:
    mark = 'SIG ' if r['significant'] else '    '
    print(f"{mark}{r['delta']:+.4f}  [{r['lo']:+.4f}, {r['hi']:+.4f}]  {r['A']}  ->  {r['B']}")

Path('paired_significance.json').write_text(
    json.dumps({'seed': SEED, 'samples': BOOTSTRAP_SAMPLES,
                'metric': 'retrieval.ndcg@5', 'comparisons': rows}, indent=2) + '\n',
    encoding='utf-8')
print('\nDa ghi paired_significance.json')


---
## 5. Cách đưa vào báo cáo

Với mỗi so sánh, thay câu chữ theo kết quả:

| Kết quả | Câu được phép viết |
|---|---|
| CI của hiệu **không chứa 0** | *"A hơn B có ý nghĩa thống kê (paired bootstrap, n=281, seed 42)"* |
| CI của hiệu **chứa 0** | *"dữ liệu không tách được A và B; lựa chọn dựa trên cơ chế/độ trễ"* |

**Đừng bỏ kết quả bất lợi.** Việc nói rõ chỗ nào không chứng minh được là thứ
làm phần chứng minh được trở nên đáng tin. Báo cáo đã ghi nhận điều đó ở mục
3.2B (BGE-M3 vs BM25) và cần ghi nhận tương tự ở mục 3.2C nếu `e5` không
tách bạch khỏi `bge-small`.

Kết quả nằm ở `paired_significance.json` — gửi lại file này là đủ để cập nhật báo cáo.
